# CODIGO DE EJECUCIÓN

*NOTA: Para poder usar este código de entrenamiento hay que lanzarlo desde exactamente el mismo entorno en el que fue creado.*

*Se puede instalar ese entorno en la nueva máquina usando el environment.yml que creamos en el set up del proyecto*

*Copiar el proyecto1.yml que tenemos en la carpeta Documentos, al directorio y en el terminal o anaconda prompt ejecutar:*

conda env create --file proyecto1.yml --name proyecto1

In [1]:
#Importamos los paquetes necesarios
import cloudpickle
import pandas as pd
import sqlalchemy as sa
from janitor import clean_names

# PRECIO_M2 IDEALISTA
#Ruta del proyecto
ruta_proyecto = 'C:/Users/Oscar/OneDrive - FM4/Escritorio/Python Data Mastery/EstructuraDirectorio/03_MACHINE_LEARNING/08_CASOS/007_AIRBNB'

#Nombre del fichero de datos
nombre_fichero_datos = 'precios_idealista_Dic24.csv'

#Cargar los datos
ruta_completa = ruta_proyecto + '/02_Datos/01_Originales/' + nombre_fichero_datos
precio_m2 = pd.read_csv(ruta_completa)


#Pipeline con las modificaciones hechas en la tabla
nombre_pipe = 'pipe_idealista.pickle'
ruta_pipe = ruta_proyecto + '/04_Modelos/' + nombre_pipe
with open(ruta_pipe, mode='rb') as file:
    pipe_idealista = cloudpickle.load(file)

precio_m2 = pipe_idealista.transform(precio_m2)

# LISTINGS AIRBNB
#Ruta del proyecto
ruta_proyecto = 'C:/Users/Oscar/OneDrive - FM4/Escritorio/Python Data Mastery/EstructuraDirectorio/03_MACHINE_LEARNING/08_CASOS/007_AIRBNB'

con = sa.create_engine('sqlite:///C:/Users/Oscar/OneDrive - FM4/Escritorio/Python Data Mastery/EstructuraDirectorio/03_MACHINE_LEARNING/08_CASOS/007_AIRBNB/02_Datos/01_Originales/airbnb2025.db')

#Nombre del fichero de datos
nombre_fichero_datos = 'airbnb2025.db'

#Cargar los datos
ruta_completa = ruta_proyecto + '/02_Datos/01_Originales/' + nombre_fichero_datos
listings = pd.read_sql('listings', con).drop(columns='index')#.set_index('id')



# LISTINGS_DET AIRBNB
#Ruta del proyecto
ruta_proyecto = 'C:/Users/Oscar/OneDrive - FM4/Escritorio/Python Data Mastery/EstructuraDirectorio/03_MACHINE_LEARNING/08_CASOS/007_AIRBNB'

con = sa.create_engine('sqlite:///C:/Users/Oscar/OneDrive - FM4/Escritorio/Python Data Mastery/EstructuraDirectorio/03_MACHINE_LEARNING/08_CASOS/007_AIRBNB/02_Datos/01_Originales/airbnb2025.db')

#Nombre del fichero de datos
nombre_fichero_datos = 'airbnb2025.db'

#Cargar los datos
ruta_completa = ruta_proyecto + '/02_Datos/01_Originales/' + nombre_fichero_datos
listings_det = pd.read_sql('listings_det', con).drop(columns='index')#.set_index('id')


#Pipeline con las modificaciones hechas en la tabla
nombre_pipe = 'pipe_preparacion_df.pickle'
ruta_pipe = ruta_proyecto + '/04_Modelos/' + nombre_pipe
with open(ruta_pipe, mode='rb') as file:
    pipe_preparacion_df = cloudpickle.load(file)

df = pipe_preparacion_df.transform((listings_det, listings, precio_m2))

#Pipeline con las modificaciones hechas en la tabla
nombre_pipe = 'pipe_calidad_datos.pickle'
ruta_pipe = ruta_proyecto + '/04_Modelos/' + nombre_pipe
with open(ruta_pipe, mode='rb') as file:
    pipe_calidad_datos = cloudpickle.load(file)
    
df = pipe_calidad_datos.transform(df)

#Pipeline con las modificaciones hechas en la tabla
nombre_pipe = 'pipe_gestion_variables.pickle'
ruta_pipe = ruta_proyecto + '/04_Modelos/' + nombre_pipe
with open(ruta_pipe, mode='rb') as file:
    pipe_gestion_variables = cloudpickle.load(file)

df = pipe_gestion_variables.transform(df)

id_df = df['id'].copy()
id_df.to_sql('id_df', con = con, if_exists = 'replace')

precio_total_df = df[['id', 'precio_total']].copy()
precio_total_df.to_sql('precio_total_df', con = con, if_exists = 'replace')

variables_finales = [
'accommodates',
'availability_30',
'availability_365',
'availability_90',
'bedrooms',
'bed',
'calculated_host_listings_count',
'has_availability',
'host_response_rate',
'host_response_time',
'instant_bookable',
'maximum_nights',
'neighbourhood_group',
'price',
'property_type',
'room_type',
'license',
'bathrooms',
'air',
'allowed',
'cleaning',
'hot water iron',
'iron',
'microwave',
'parking',
'refrigerator',
'shampoo',
'tv washer',
'wifi kitchen',
'neighbourhood_cleansed',
'and silverware',
'body',
'clothing',
'clothing storage',
'coffee',
'elevator',
'hair',
'host_identity_verified',
'host_verifications',
'kitchen essentials',
'allowed',
'basics',
'beds',
'dryer bed',
'hair dryer',
'maker',
'microwave hangers',
'oven',
'stove',
'review_scores_rating',
'review_scores_accuracy',
'review_scores_cleanliness',
'review_scores_checkin',
'review_scores_communication',
'review_scores_location',
'review_scores_value',
'host_is_superhost',
'bathrooms_text',
'precio_m2',
'latitude',
'longitude',
'amenities',
'number_of_reviews']

df = df[variables_finales]

nombre_pipe_ejecucion = 'pipe_ejecucion.pickle'
ruta_pipe_ejecucion = ruta_proyecto + '/04_Modelos/' + nombre_pipe_ejecucion
with open(ruta_pipe_ejecucion, mode='rb') as file:
   pipe_ejecucion = cloudpickle.load(file)


scoring = pipe_ejecucion.predict_proba(df)[:, 1]

In [2]:
scoring

array([0.9977684 , 1.        , 0.99999976, ..., 0.00968286, 0.01205544,
       1.        ], dtype=float32)

Este es el código final de producción que pasaremos al informático para su aplicación. 

Podemos pasarlo en formato notebook o formato Python descargándolo en 'File/Download as/Python (.py)'